In [2]:
from datasets import load_dataset

funsd = load_dataset("nielsr/funsd")

FUNSD_LABELS = ["O", "question", "answer", "header", "other"]

def convert_funsd_example(example):
    boxes = []
    words = example["words"]
    bboxes = example["bboxes"]
    ner_tags = example["ner_tags"]
    
    for word, box, tag in zip(words, bboxes, ner_tags):
        boxes.append({
            "box": box,
            "label": FUNSD_LABELS[tag] if tag < len(FUNSD_LABELS) else "other",
            "text": word
        })
    return boxes

example = funsd['test'][0]
gt_boxes = convert_funsd_example(example)
gt_boxes[:5]

c:\Users\Acer\miniconda3\envs\docuagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Acer\miniconda3\envs\docuagent\Lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\datasets--nielsr--funsd. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate deve

[{'box': [135, 345, 171, 359], 'label': 'header', 'text': 'TO:'},
 {'box': [135, 406, 194, 423], 'label': 'header', 'text': 'DATE:'},
 {'box': [611, 440, 629, 455], 'label': 'other', 'text': '3'},
 {'box': [330, 84, 363, 98], 'label': 'header', 'text': 'Fax:'},
 {'box': [136, 691, 184, 708], 'label': 'header', 'text': 'NOTE:'}]

In [4]:
import json
from datasets import load_dataset
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

funsd = load_dataset("nielsr/funsd")
test_set = funsd['test']
train_set = funsd['train']
OUTPUT_DIR = Path("C:/Users/Legion/Desktop/DocuAgent-AP/outputs")
EVAL_DIR = Path("evaluation")
EVAL_DIR.mkdir(exist_ok=True)

metrics = []

def get_gt_text(example):
    words = [w for w in example['words'] if w.strip() != ""]
    return " ".join(words)

def get_pred_text(result_json):
    pred_texts = []
    for box in result_json.get("page_1", {}).get("boxes", []):
        content = box.get("block_content")
        if content:
            if isinstance(content, dict) and "base_64_encoded" in content:
                continue
            pred_texts.append(str(content))
    return " ".join(pred_texts)

def tokenize(text):
    return text.split()

def compute_metrics(gt_text, pred_text):
    gt_tokens = tokenize(gt_text)
    pred_tokens = tokenize(pred_text)
    
    all_tokens = list(set(gt_tokens + pred_tokens))
    gt_vec = [1 if t in gt_tokens else 0 for t in all_tokens]
    pred_vec = [1 if t in pred_tokens else 0 for t in all_tokens]
    
    precision = precision_score(gt_vec, pred_vec)
    recall = recall_score(gt_vec, pred_vec)
    f1 = f1_score(gt_vec, pred_vec)
    
    return precision, recall, f1

In [5]:
metrics = []

for img_path in sorted(EVAL_DIR.glob("*.png")):
    img_name = img_path.stem
    is_train = "_train" in img_name
    index = int(img_name.replace("_train", ""))

    example = train_set[index] if is_train else test_set[index]
    gt_text = get_gt_text(example)

    if is_train:
        result_json_path = (OUTPUT_DIR / f"{index}_train" / f"{index}_train_all_results.json").resolve()
    else:
        result_json_path = (OUTPUT_DIR / str(index) / f"{index}_all_results.json").resolve()

    if not result_json_path.exists():
        print(f"[⚠️ Missing JSON] {result_json_path}")
        continue

    with open(result_json_path, "r", encoding="utf-8") as f:
        pipeline_pred = json.load(f)

    pred_text = get_pred_text(pipeline_pred)
    if not pred_text.strip():
        continue

    precision, recall, f1 = compute_metrics(gt_text, pred_text)
    metrics.append((img_name, precision, recall, f1))
    print(f"{img_name}: P={precision:.2f}, R={recall:.2f}, F1={f1:.2f}")

arr = np.array([m[1:] for m in metrics])
print(f"\nAverage Precision={arr[:,0].mean():.2f}, Recall={arr[:,1].mean():.2f}, F1={arr[:,2].mean():.2f}")


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [6]:
import sys, os
proj = r"C:\Users\Acer\Desktop\DocuAgent-AP"
if proj not in sys.path:
    sys.path.insert(0, proj)
os.chdir(proj)
print("cwd:", os.getcwd())

cwd: C:\Users\Acer\Desktop\DocuAgent-AP


In [8]:
from pathlib import Path
from src.pipeline.pdf_ingest import process_pdf_pipeline
from src.utils.paths import DATA, OUTPUT

fname = "certificatecer.pdf"   # change to the uploaded filename
pdf_path = Path(DATA) / fname
print("Will run on:", pdf_path)
process_pdf_pipeline(pdf_path=str(pdf_path), layout_result_save=True, preprocess=False)
print("Finished. Output folder:", OUTPUT / pdf_path.stem)

Will run on: C:\Users\Acer\Desktop\DocuAgent-AP\data\certificatecer.pdf
[INFO] PDF opened successfully.
[INFO] Converting PDF to images for certificatecer


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Acer\.paddlex\official_models\PP-DocLayout_plus-L`.


[INFO] Processing page 1 of certificatecer

[INFO] Starting pipeline for: certificatecer_page_1
[INFO] Output directory: C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer
[STEP 2/6] Detecting layout...
[INFO] Initializing PP-DocLayout model...


C:\Users\Acer\Desktop\DocuAgent-AP\src\models\paddle_models.py:106: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  _ocr_instance = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Acer\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.


[INFO] Layout Detection Time for certificatecer_page_1 : 2.92s
[STEP 3/6] Masking non-text regions...
[INFO] Masking Time for certificatecer_page_1 : 0.01s
[STEP 4/6] Running OCR...
[INFO] Initializing PaddleOCR model...


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Acer\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Acer\.paddlex\official_models\PP-OCRv5_server_rec`.


[INFO] Performing OCR on detected regions for certificatecer_page_1
[INFO] OCR on certificatecer_page_1 took 4.24 seconds
[STEP 5/6] Processing detected regions...

 [WARNING] No tables detected or extracted in this image
 [WARNING] No charts detected or extracted in this image
[STEP 6/6] Saving all results...

[SAVE] Saving results to: C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer
[INFO] Preprocessed image saved → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\preprocessed\certificatecer_page_1_preprocessed.png
[INFO] Masked image saved → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\masked\certificatecer_page_1_masked.png
[INFO] Layout image saved → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\layout\img\certificatecer_page_1_layout.png


The result has multiple img files need to be saved. But the `save_path` has been specified as `C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\ocr\image\certificatecer_page_1_ocr.png`!


[INFO] OCR visualization saved → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\ocr\image\certificatecer_page_1_ocr.png
[INFO] OCR results saved for page 1 → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\ocr\json\certificatecer_ocr.json
[INFO] Complete results saved for page 1 → C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer\certificatecer_all_results.json
[SAVE] All results saved successfully


 [SUCCESS] Pipeline completed for: certificatecer_page_1
  [TOTAL TIME FOR IMAGE] 61.14s

[INFO] Processing page 1 took 61.16 seconds
[INFO] PDF to image conversion took 61.71 seconds
[INFO] Pipeline on certificatecer took 61.83 seconds
Finished. Output folder: C:\Users\Acer\Desktop\DocuAgent-AP\outputs\certificatecer


In [9]:
import json
from pathlib import Path

def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i] + [0] * lb
        for j, cb in enumerate(b, start=1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (0 if ca == cb else 1)
            cur[j] = min(ins, dele, sub)
        prev = cur
    return prev[lb]

In [10]:
def extract_text_from_result(json_path):
    p = Path(json_path)
    data = json.loads(p.read_text(encoding="utf-8"))
    parts = []
    # handle structure like {"page_1": {"boxes":[{"block_content": "..."}]}}
    for key in sorted(data.keys()):
        page = data[key]
        if isinstance(page, dict):
            boxes = page.get("boxes") or page.get("blocks") or []
            if isinstance(boxes, list):
                for b in boxes:
                    if isinstance(b, dict):
                        text = b.get("block_content") or b.get("text") or b.get("content")
                        if text:
                            parts.append(str(text))
            else:
                parts.append(str(page))
        else:
            parts.append(str(page))
    return "\n".join(parts).strip()

In [13]:
pred = Path("outputs") / "certificatecer" / "certificatecer_all_results.json"
gt = Path("data") / "certificatecer_groundtruth.txt"
pred_text = extract_text_from_result(pred)
gt_text = gt.read_text(encoding="utf-8").strip()
dist = levenshtein(pred_text, gt_text)
cer = dist / len(gt_text) if len(gt_text) else None
print("CER:", cer, "edits:", dist, "chars:", len(gt_text))

CER: 0.017804154302670624 edits: 6 chars: 337
